# LLM-Only Model Download
This notebook downloads only the Qwen3-4B-Instruct-2507 GGUF model in Q4_K_M quantization. It does not download a VLM or vision projector.

## Cell 1 – Install Hugging Face Hub

In [ ]:
!pip install -q -U huggingface_hub

## Cell 2 – Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 – Download the LLM

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

BASE = "/content/drive/MyDrive/CivicConnect_Models/LLM"
REPO = "bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF"
QUANTIZATION = "q4_k_m"

os.makedirs(BASE, exist_ok=True)
api = HfApi()
repo_files = api.list_repo_files(REPO)

matches = [
    name for name in repo_files
    if name.lower().endswith(".gguf")
    and QUANTIZATION in name.lower()
]

if not matches:
    raise FileNotFoundError(
        f"No {QUANTIZATION.upper()} GGUF file found in {REPO}"
    )

# Prefer a non-split model file when available.
single_files = [
    name for name in matches
    if "-00001-of-" not in name.lower()
]
model_file = single_files[0] if single_files else matches[0]

print("Downloading:", model_file)
model_path = hf_hub_download(
    repo_id=REPO,
    filename=model_file,
    local_dir=BASE
)

print("\nLLM download completed.")
print("Saved to:", model_path)
print(f"Size: {os.path.getsize(model_path) / (1024**3):.2f} GB")

## Cell 4 – Verify the downloaded model

In [ ]:
for root, _, files in os.walk(BASE):
    for filename in files:
        if filename.lower().endswith(".gguf"):
            file_path = os.path.join(root, filename)
            size_gb = os.path.getsize(file_path) / (1024 ** 3)
            print(f"{filename} | {size_gb:.2f} GB")

## Cell 5 – Create a ZIP file (optional)

In [ ]:
import shutil

zip_path = "/content/LLM_Model.zip"
shutil.make_archive(
    "/content/LLM_Model",
    "zip",
    BASE
)
print("ZIP created:", zip_path)